# 🌐 Data Collection & Web Scraping — Complete Masterclass

> **Days 15-17 Transcripts**: HTML architecture, HTTP requests, REST APIs, BeautifulSoup tag traversal, attributes, CSS selectors aur tabular data pipeline.

### 🗺️ Topics Index:
1. **HTML Architecture & Elements** (`div`, `class`, `id`)
2. **HTTP Requests Basics** (`requests.get`, Status Codes)
3. **HTTP Headers & User-Agents** (Anti-blocking)
4. **REST APIs & JSON Parsing** (`requests.get().json()`)
5. **API to DataFrame Normalization** (`pd.json_normalize`)
6. **BeautifulSoup Tag Extraction** (`find`, `find_all`)
7. **CSS Selectors** (`select`, `select_one`)
8. **DOM Tree Traversal** (`parent`, `children`, `siblings`)
9. **Multi-Element Real Scraper** (Country / E-Commerce Cards)
10. **Storage Pipeline** (Export Scraped Data to CSV)


### Operation 1: HTML Architecture & Document Structure
- **Tags & Attributes:** `<div class="container" id="main">` me `class` styling ke liye aur `id` unique identifier ke liye hoti hai.


In [1]:
sample_html = """
<!DOCTYPE html>
<html>
<head><title>Scraping Demo Store</title></head>
<body>
    <div id="main-content" class="container">
        <h1 class="store-title">AI Powered Electronics</h1>
        <div class="product-card" data-id="101">
            <h3 class="product-name">Neural Laptop Pro</h3>
            <span class="price">$1299.99</span>
            <span class="stock-status in-stock">Available</span>
            <a href="/products/101" class="buy-link">View Details</a>
        </div>
        <div class="product-card" data-id="102">
            <h3 class="product-name">Quantum Watch 2</h3>
            <span class="price">$349.50</span>
            <span class="stock-status out-of-stock">Sold Out</span>
            <a href="/products/102" class="buy-link">View Details</a>
        </div>
    </div>
</body>
</html>
"""
print("Sample HTML Document Defined (Length:", len(sample_html), "characters)")


Sample HTML Document Defined (Length: 826 characters)


### Operation 2: HTTP Requests Basics (`requests.get`)
- **Status Codes:** `200` OK, `404` Not Found, `403` Forbidden, `500` Server Error.
- `response.text` raw string return karta hai, `response.json()` dictionary me decode karta hai.


In [2]:
import requests

url = "https://httpbin.org/get"
try:
    response = requests.get(url, timeout=5)
    print("HTTP Status Code:", response.status_code)
    print("Content-Type:    ", response.headers.get("Content-Type"))
    print("Request Success: ", response.status_code == 200)
except Exception as e:
    print("Offline Simulation: HTTP GET Status 200 OK (Connection Mocked)")


HTTP Status Code: 200
Content-Type:     application/json
Request Success:  True


### Operation 3: HTTP Headers & User-Agent (Anti-blocking)
- Servers generic Python bots ko block kar dete hain; browser-like `User-Agent` pass karna zaroori hai.


In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

print("Configured Custom Headers:")
for k, v in headers.items():
    print(f"  {k}: {v}")


Configured Custom Headers:
  User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36
  Accept-Language: en-US,en;q=0.9


### Operation 4: REST APIs & JSON Parsing
- APIs structured data exchange provide karti hain bina HTML parse kiye.


In [4]:
import pandas as pd

api_url = "https://stephen-king-api.onrender.com/api/books"
try:
    api_res = requests.get(api_url, timeout=5)
    api_res.raise_for_status()
    json_data = api_res.json()
    print("API Data Loaded Online! Total Records:", len(json_data.get("data", [])))
except Exception:
    # Guaranteed Offline Fallback Data
    json_data = {
        "data": [
            {"id": 1, "Title": "The Shining", "Year": 1977, "ISBN": "978-0385121675", "Pages": 447},
            {"id": 2, "Title": "It", "Year": 1986, "ISBN": "978-0670813025", "Pages": 1138},
            {"id": 3, "Title": "Misery", "Year": 1987, "ISBN": "978-0670813643", "Pages": 310},
            {"id": 4, "Title": "Carrie", "Year": 1974, "ISBN": "978-0385086950", "Pages": 199},
            {"id": 5, "Title": "The Stand", "Year": 1978, "ISBN": "978-0385121682", "Pages": 823}
        ]
    }
    print("API Data Loaded from Offline Fallback (5 Book Records)")


API Data Loaded Online! Total Records: 68


### Operation 5: API to DataFrame Normalization (`pd.json_normalize`)
- Nested JSON payloads ko direct flat 2D tabular DataFrame me convert karta hai.


In [5]:
df_books = pd.json_normalize(json_data["data"])
df_books_clean = df_books[['id', 'Title', 'Year', 'ISBN']].head(5)

print("--- Clean Books DataFrame from API ---")
display(df_books_clean)


--- Clean Books DataFrame from API ---


,id,Title,Year,ISBN
0,1,Carrie,1974,978-0-385-08695-0
1,2,Salem's Lot,1975,978-0-385-00751-1
2,3,The Shining,1977,978-0-385-12167-5
3,4,Rage,1977,978-0-451-07645-8
4,5,The Stand,1978,978-0-385-12168-2


### Operation 6: BeautifulSoup Tag Extraction (`find`, `find_all`)
- **`find(tag, class_)`:** Pehla matching element.
- **`find_all(tag)`:** Saare matching elements ki list.


In [6]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(sample_html, "html.parser")

# 1. Page Title
title_tag = soup.find("title")
print("Page Title Tag:", title_tag.get_text(strip=True))

# 2. Main Heading
h1_tag = soup.find("h1", class_="store-title")
print("Main Heading:   ", h1_tag.get_text(strip=True))

# 3. All Product Names
all_products = soup.find_all("h3", class_="product-name")
print(f"\nFound {len(all_products)} products:")
for p in all_products:
    print("  *", p.get_text(strip=True))


Page Title Tag: Scraping Demo Store
Main Heading:    AI Powered Electronics

Found 2 products:
  * Neural Laptop Pro
  * Quantum Watch 2


### Operation 7: CSS Selectors (`select`, `select_one`)
- CSS selector syntax: `.class_name`, `#id_name`, `tag.class`.


In [7]:
# 1. select_one
product_name = soup.select_one("h3.product-name").get_text(strip=True)
product_price = soup.select_one("span.price").get_text(strip=True)

print("First Product via CSS Selector:")
print("  Name: ", product_name)
print("  Price:", product_price)

# 2. select (All matches)
in_stock_items = soup.select(".product-card:has(.in-stock) .product-name")
print(f"\nIn-Stock Items ({len(in_stock_items)}):")
for item in in_stock_items:
    print("  -", item.get_text(strip=True))


First Product via CSS Selector:
  Name:  Neural Laptop Pro
  Price: $1299.99

In-Stock Items (1):
  - Neural Laptop Pro


### Operation 8: DOM Tree Traversal (`parent`, `siblings`)
- Tag hierarchy me navigate karna: `.find_parent()`, `.find_next_sibling()`.


In [8]:
h3_tag = soup.find("h3")
print("Target Tag:      ", h3_tag.get_text(strip=True))

# Parent
parent_div = h3_tag.find_parent("div")
print("Parent Div Class:", parent_div["class"])
print("Data ID Attribute:", parent_div.get("data-id"))

# Next Sibling
price_sibling = h3_tag.find_next_sibling("span")
print("Next Sibling Price:", price_sibling.get_text(strip=True))


Target Tag:       Neural Laptop Pro
Parent Div Class: ['product-card']
Data ID Attribute: 101
Next Sibling Price: $1299.99


### Operation 9: Multi-Element Structured Extraction
- Multiple cards se structured dictionary list extract karna.


In [9]:
product_cards = soup.find_all("div", class_="product-card")
catalog = []

for card in product_cards:
    name = card.find("h3", class_="product-name").get_text(strip=True)
    price = card.find("span", class_="price").get_text(strip=True)
    status = card.find("span", class_="stock-status").get_text(strip=True)
    link = card.find("a", class_="buy-link")["href"]
    
    catalog.append({
        "Product": name,
        "Price_USD": float(price.replace("$", "")),
        "Status": status,
        "URL": link
    })

print(f"Successfully parsed {len(catalog)} products into dictionary list:")
for item in catalog:
    print(" ", item)


Successfully parsed 2 products into dictionary list:
  {'Product': 'Neural Laptop Pro', 'Price_USD': 1299.99, 'Status': 'Available', 'URL': '/products/101'}
  {'Product': 'Quantum Watch 2', 'Price_USD': 349.5, 'Status': 'Sold Out', 'URL': '/products/102'}


### Operation 10: Storage Pipeline — Exporting Scraped Data
- Parsed records ko DataFrame me load karke clean CSV file me save karna.


In [10]:
df_catalog = pd.DataFrame(catalog)
print("--- Final Scraped Catalog DataFrame ---")
display(df_catalog)

# Export to CSV
csv_filename = "scraped_products.csv"
df_catalog.to_csv(csv_filename, index=False)

import os
print(f"\nSaved to '{csv_filename}' | Exists: {os.path.exists(csv_filename)}")


--- Final Scraped Catalog DataFrame ---


,Product,Price_USD,Status,URL
0,Neural Laptop Pro,1299.99,Available,/products/101
1,Quantum Watch 2,349.50,Sold Out,/products/102



Saved to 'scraped_products.csv' | Exists: True


---

## 🎯 Days 15-17 Summary & Quick Revision Checklist
- [x] **HTML Architecture:** Tags, attributes, classes & IDs.
- [x] **HTTP Requests:** `requests.get()`, status codes, headers.
- [x] **APIs:** REST API JSON consumption & `pd.json_normalize()`.
- [x] **BeautifulSoup:** `find()`, `find_all()`, `.get_text()`, `.attrs`.
- [x] **CSS Selectors:** `.select()` and `.select_one()`.
- [x] **DOM Traversal:** `find_parent()`, `find_next_sibling()`.
- [x] **Storage Pipeline:** DataFrame transformation to `.to_csv()`.

---
### 🏆 Web Scraping & Data Collection Masterclass Completed!
